In [6]:
# 1. Setup configuration parameters.

from pathlib import Path
from time import perf_counter

import numpy as np

from ramsey import (
    REdgeViolationPolicy,
    REnvironment,
    REnvironmentConfig,
    RGraph,
    RGreedyPolicy,
    RMonochromaticObjective,
    RProblem,
    RSearch,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)


RANDOM_SEED = 202_608_060

N_VERTICES = 43
NUMBER_OF_RUNS = 10

ARCHIVE_MAXIMUM_SCORE = 399

MAX_STEPS = 500

EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000


project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)


problem = RProblem.r55(
    n_vertices=N_VERTICES,
)

graph = RGraph(problem)

archive = RSQLiteArchive(
    DATABASE_PATH
)

rng = np.random.default_rng(
    RANDOM_SEED
)

print("Archive best:", archive.best_score(graph))

Archive best: 278


In [7]:
# 2. Select the fixed seeds.

eligible_records = (
    archive.colorings_in_score_range(
        maximum_score=ARCHIVE_MAXIMUM_SCORE,
        graph=graph,
    )
)

if len(eligible_records) < NUMBER_OF_RUNS:
    raise RuntimeError(
        "Not enough eligible archive colorings."
    )

selected_indices = rng.choice(
    len(eligible_records),
    size=NUMBER_OF_RUNS,
    replace=False,
)

seed_records = [
    eligible_records[int(index)]
    for index in selected_indices
]

print(
    "Eligible archive colorings:",
    len(eligible_records),
)

print(
    "Selected seed scores:",
    [record.score for record in seed_records],
)

Eligible archive colorings: 904
Selected seed scores: [335, 311, 301, 341, 334, 305, 350, 314, 312, 304]


In [8]:
# 3. Build the competing searches

def make_environment() -> REnvironment:
    return REnvironment(
        graph=graph,
        objective=RMonochromaticObjective(),
        memory=RTabuMemory(
            graph.number_of_edges,
            RTabuMemoryConfig(
                edge_tenure=EDGE_TABU_TENURE,
                visited_state_window=(
                    VISITED_STATE_WINDOW
                ),
            ),
        ),
        config=REnvironmentConfig(
            max_steps=MAX_STEPS,
        ),
    )


violation_policy = REdgeViolationPolicy.REdgeViolationPolicy(
    np.random.default_rng(
        RANDOM_SEED + 1
    )
)

greedy_policy = RGreedyPolicy(
    np.random.default_rng(
        RANDOM_SEED + 2
    ),
    use_objective_reward=False,
)


violation_search = RSearch(
    make_environment(),
    violation_policy,
)

greedy_search = RSearch(
    make_environment(),
    greedy_policy,
)

In [9]:
# 4. Run the paired experiment.

results = []

experiment_start = perf_counter()


for run_number, record in enumerate(
    seed_records
):
    archived = archive.load_coloring(
        record.coloring_id,
        graph,
    )

    seed_coloring = archived.coloring


    # --------------------------------------------
    # Violation-load policy
    # --------------------------------------------

    start = perf_counter()

    violation_result = (
        violation_search.run(
            seed_coloring
        )
    )

    violation_elapsed = (
        perf_counter() - start
    )


    # --------------------------------------------
    # Ordinary exact greedy policy
    # --------------------------------------------

    start = perf_counter()

    greedy_result = (
        greedy_search.run(
            seed_coloring
        )
    )

    greedy_elapsed = (
        perf_counter() - start
    )


    results.append(
        {
            "archive_id": record.coloring_id,
            "initial": record.score,

            "violation_final": (
                violation_result.final_score
            ),
            "violation_best": (
                violation_result.best_score
            ),
            "violation_time": (
                violation_elapsed
            ),

            "greedy_final": (
                greedy_result.final_score
            ),
            "greedy_best": (
                greedy_result.best_score
            ),
            "greedy_time": (
                greedy_elapsed
            ),
            "violation_best_coloring": (
                violation_result.best_coloring
            ),
            "greedy_best_coloring": (
                greedy_result.best_coloring
            ),
        }
    )


    print(
        f"Run {run_number:2d} | "
        f"id={record.coloring_id:5d} | "
        f"initial={record.score:4d} | "
        f"violation="
        f"{violation_result.best_score:4d} | "
        f"greedy="
        f"{greedy_result.best_score:4d}"
    )


experiment_elapsed = (
    perf_counter()
    - experiment_start
)

print()
print(
    "Total time:",
    f"{experiment_elapsed:.2f}s",
)

Run  0 | id= 1463 | initial= 335 | violation= 335 | greedy= 169
Run  1 | id= 1580 | initial= 311 | violation= 311 | greedy= 170
Run  2 | id= 1567 | initial= 301 | violation= 298 | greedy= 165
Run  3 | id= 1177 | initial= 341 | violation= 331 | greedy= 185
Run  4 | id= 1511 | initial= 334 | violation= 334 | greedy= 181
Run  5 | id= 1668 | initial= 305 | violation= 305 | greedy= 172
Run  6 | id= 1202 | initial= 350 | violation= 344 | greedy= 181
Run  7 | id= 1691 | initial= 314 | violation= 314 | greedy= 185
Run  8 | id= 1811 | initial= 312 | violation= 308 | greedy= 180
Run  9 | id= 1554 | initial= 304 | violation= 296 | greedy= 175

Total time: 30.34s


In [10]:
# 5. Compare results.

initial_scores = np.asarray(
    [
        result["initial"]
        for result in results
    ]
)

violation_best = np.asarray(
    [
        result["violation_best"]
        for result in results
    ]
)

greedy_best = np.asarray(
    [
        result["greedy_best"]
        for result in results
    ]
)


violation_wins = int(
    np.count_nonzero(
        violation_best < greedy_best
    )
)

greedy_wins = int(
    np.count_nonzero(
        greedy_best < violation_best
    )
)

ties = int(
    np.count_nonzero(
        greedy_best == violation_best
    )
)


print("Runs:", len(results))
print()

print(
    "Mean initial:",
    f"{initial_scores.mean():.2f}",
)

print()

print("Violation-load policy")
print(
    "  Mean best:",
    f"{violation_best.mean():.2f}",
)
print(
    "  Minimum:",
    violation_best.min(),
)
print(
    "  Mean reduction:",
    f"{(initial_scores - violation_best).mean():.2f}",
)

print()

print("Greedy exact-score policy")
print(
    "  Mean best:",
    f"{greedy_best.mean():.2f}",
)
print(
    "  Minimum:",
    greedy_best.min(),
)
print(
    "  Mean reduction:",
    f"{(initial_scores - greedy_best).mean():.2f}",
)

print()

print("Head-to-head")
print("  Violation wins:", violation_wins)
print("  Greedy wins:", greedy_wins)
print("  Ties:", ties)

Runs: 10

Mean initial: 320.70

Violation-load policy
  Mean best: 317.60
  Minimum: 296
  Mean reduction: 3.10

Greedy exact-score policy
  Mean best: 176.30
  Minimum: 165
  Mean reduction: 144.40

Head-to-head
  Violation wins: 0
  Greedy wins: 10
  Ties: 0
